In [ ]:
# Cleans Bronze data and upserts current records into Silver Delta tables.

import json, os
from pyspark.sql.functions import col, upper, trim, to_date, to_timestamp, coalesce, lit, row_number, max as spark_max, year, month, date_format, dayofmonth, weekofyear, dayofweek
from pyspark.sql.window import Window

BASE = "/lakehouse/default/Files"
WATERMARK_PATH = f"{BASE}/bronze/_watermark/watermark.json"


def parse_ts(column_name):
    return coalesce(
        to_timestamp(col(column_name), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
        to_timestamp(col(column_name), "M/d/yyyy h:mm a"),
        col(column_name).cast("timestamp")
    )


def table_exists(table_name):
    return spark.catalog.tableExists(table_name)


def latest_by_key(df, key_col):
    w = Window.partitionBy(key_col).orderBy(col("last_modified_utc").desc())
    return df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

# Bronze tables expected from pipeline/static ingest.
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_terminal_raw (
    terminal_code STRING, terminal_name STRING, country STRING, port_hub STRING,
    total_capacity_cbm STRING, number_of_tanks STRING, operational_since STRING,
    specialty STRING, source_url STRING, last_modified_utc STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_product_raw (
    product_code STRING, product_name STRING, product_family STRING, is_new_energy STRING,
    typical_density_kg_m3 STRING, last_modified_utc STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_contract_raw (
    contract_id STRING, customer_code STRING, customer_name STRING, customer_type STRING,
    customer_country STRING, terminal_code STRING, product_code STRING,
    contract_start_date STRING, contract_end_date STRING, reserved_capacity_cbm STRING,
    storage_fee_eur_per_cbm_month STRING, handling_fee_eur_per_cbm STRING,
    contract_status STRING, last_modified_utc STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_movement_raw (
    movement_id STRING, movement_datetime_utc STRING, planned_datetime_utc STRING,
    terminal_code STRING, product_code STRING, contract_id STRING, movement_type STRING,
    transport_mode STRING, volume_cbm STRING, movement_status STRING, delay_hours STRING,
    delay_reason STRING, last_modified_utc STRING
) USING DELTA
""")

# Silver table definitions.
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_terminal (
    terminal_code STRING, terminal_name STRING, country STRING, port_hub STRING,
    total_capacity_cbm INT, number_of_tanks INT, operational_since INT,
    specialty STRING, source_url STRING, last_modified_utc TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver_product (
    product_code STRING, product_name STRING, product_family STRING, is_new_energy STRING,
    typical_density_kg_m3 INT, last_modified_utc TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver_contract (
    contract_id STRING, customer_code STRING, customer_name STRING, customer_type STRING,
    customer_country STRING, terminal_code STRING, product_code STRING,
    contract_start_date DATE, contract_end_date DATE, reserved_capacity_cbm INT,
    storage_fee_eur_per_cbm_month DOUBLE, handling_fee_eur_per_cbm DOUBLE,
    contract_status STRING, last_modified_utc TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver_movement (
    movement_id STRING, movement_datetime_utc TIMESTAMP, planned_datetime_utc TIMESTAMP,
    terminal_code STRING, product_code STRING, contract_id STRING, movement_type STRING,
    transport_mode STRING, volume_cbm DOUBLE, movement_status STRING, delay_hours DOUBLE,
    delay_reason STRING, last_modified_utc TIMESTAMP, movement_date DATE, is_completed INT,
    is_delayed INT, estimated_handling_fee_eur DOUBLE
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver_date (
    date_key DATE, year_number INT, month_number INT, month_name STRING,
    day_of_month INT, iso_week INT, day_of_week_number INT
) USING DELTA
""")

# Basic cleaning and latest-row selection.
terminal = spark.table("bronze_terminal_raw").dropna(subset=["terminal_code"]) \
    .withColumn("terminal_code", upper(trim(col("terminal_code")))) \
    .withColumn("total_capacity_cbm", col("total_capacity_cbm").cast("int")) \
    .withColumn("number_of_tanks", col("number_of_tanks").cast("int")) \
    .withColumn("operational_since", col("operational_since").cast("int")) \
    .withColumn("last_modified_utc", parse_ts("last_modified_utc"))
latest_by_key(terminal, "terminal_code").createOrReplaceTempView("src_terminal")

product = spark.table("bronze_product_raw").dropna(subset=["product_code"]) \
    .withColumn("product_code", upper(trim(col("product_code")))) \
    .withColumn("typical_density_kg_m3", col("typical_density_kg_m3").cast("int")) \
    .withColumn("last_modified_utc", parse_ts("last_modified_utc"))
latest_by_key(product, "product_code").createOrReplaceTempView("src_product")

contract = spark.table("bronze_contract_raw").dropna(subset=["contract_id"]) \
    .withColumn("contract_id", upper(trim(col("contract_id")))) \
    .withColumn("terminal_code", upper(trim(col("terminal_code")))) \
    .withColumn("product_code", upper(trim(col("product_code")))) \
    .withColumn("contract_start_date", to_date(col("contract_start_date"))) \
    .withColumn("contract_end_date", to_date(col("contract_end_date"))) \
    .withColumn("reserved_capacity_cbm", col("reserved_capacity_cbm").cast("int")) \
    .withColumn("storage_fee_eur_per_cbm_month", col("storage_fee_eur_per_cbm_month").cast("double")) \
    .withColumn("handling_fee_eur_per_cbm", col("handling_fee_eur_per_cbm").cast("double")) \
    .withColumn("last_modified_utc", parse_ts("last_modified_utc"))
latest_by_key(contract, "contract_id").createOrReplaceTempView("src_contract")

movement = spark.table("bronze_movement_raw").dropna(subset=["movement_id"]) \
    .withColumn("movement_id", upper(trim(col("movement_id")))) \
    .withColumn("terminal_code", upper(trim(col("terminal_code")))) \
    .withColumn("product_code", upper(trim(col("product_code")))) \
    .withColumn("contract_id", upper(trim(col("contract_id")))) \
    .withColumn("movement_datetime_utc", parse_ts("movement_datetime_utc")) \
    .withColumn("planned_datetime_utc", parse_ts("planned_datetime_utc")) \
    .withColumn("last_modified_utc", parse_ts("last_modified_utc")) \
    .withColumn("volume_cbm", col("volume_cbm").cast("double")) \
    .withColumn("delay_hours", col("delay_hours").cast("double")) \
    .withColumn("movement_date", to_date(col("movement_datetime_utc"))) \
    .withColumn("is_completed", (upper(col("movement_status")) == "COMPLETED").cast("int")) \
    .withColumn("is_delayed", (col("delay_hours") > 0).cast("int")) \
    .withColumn("estimated_handling_fee_eur", col("volume_cbm") * lit(0.35))
latest_by_key(movement, "movement_id").createOrReplaceTempView("src_movement")

# Upsert into Silver current tables.
spark.sql("""
MERGE INTO silver_terminal t USING src_terminal s ON t.terminal_code = s.terminal_code
WHEN MATCHED AND s.last_modified_utc >= t.last_modified_utc THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
MERGE INTO silver_product t USING src_product s ON t.product_code = s.product_code
WHEN MATCHED AND s.last_modified_utc >= t.last_modified_utc THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
MERGE INTO silver_contract t USING src_contract s ON t.contract_id = s.contract_id
WHEN MATCHED AND s.last_modified_utc >= t.last_modified_utc THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
MERGE INTO silver_movement t USING src_movement s ON t.movement_id = s.movement_id
WHEN MATCHED AND s.last_modified_utc >= t.last_modified_utc THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# Date dimension.
dates = spark.sql("SELECT explode(sequence(to_date('2026-01-01'), to_date('2026-12-31'), interval 1 day)) AS date_key")
dates = dates.select(
    col("date_key"), year(col("date_key")).alias("year_number"), month(col("date_key")).alias("month_number"),
    date_format(col("date_key"), "MMMM").alias("month_name"), dayofmonth(col("date_key")).alias("day_of_month"),
    weekofyear(col("date_key")).alias("iso_week"), dayofweek(col("date_key")).alias("day_of_week_number")
)
dates.createOrReplaceTempView("src_date")
spark.sql("""
MERGE INTO silver_date t USING src_date s ON t.date_key = s.date_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# Update watermark after Silver succeeds.
wm = {}
for name, table, col_name in [
    ("dim_terminal", "silver_terminal", "last_modified_utc"),
    ("dim_product", "silver_product", "last_modified_utc"),
    ("customer_contracts", "silver_contract", "last_modified_utc"),
    ("terminal_movements", "silver_movement", "last_modified_utc")
]:
    value = spark.table(table).agg(spark_max(col(col_name))).collect()[0][0]
    wm[name] = value.strftime("%Y-%m-%dT%H:%M:%SZ") if value else "1900-01-01T00:00:00Z"

os.makedirs(os.path.dirname(WATERMARK_PATH), exist_ok=True)
with open(WATERMARK_PATH, "w") as f:
    json.dump(wm, f, indent=2)

print(json.dumps(wm, indent=2))